# 01 · Thu thập dữ liệu VSL

Chọn **lớp** và **số mẫu** cần quay ở ô cấu hình bên dưới, rồi chạy ô thu thập. Một cửa sổ OpenCV sẽ hiện ra.

| Phím | Tác dụng |
|---|---|
| `SPACE` | bắt đầu (đếm ngược → quay `RECORD_SECONDS` giây) |
| `ENTER` | lưu mẫu vừa quay (chỉ khi chất lượng đạt) |
| `F` | lưu dù chất lượng thấp |
| `R` | bỏ mẫu vừa quay |
| `U` | xoá mẫu vừa lưu gần nhất (khi đang ở trạng thái sẵn sàng) |
| `N` | sang lớp kế tiếp |
| `Q` | thoát |

Chạy bằng Jupyter **cục bộ** trên máy có webcam (cửa sổ OpenCV không hiện trên Colab/remote).
Mỗi mẫu lưu **đặc trưng thô theo từng frame** (có thể chứa NaN khi mất tay) + video gốc + `meta.json`.

In [1]:
import sys, time, unicodedata
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np

# tìm thư mục gốc dự án (chứa `src/`) và cho phép import
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.builtin.mediapipe_builtin import LandmarkerLoader, VisionRunningMode
from src.builtin.mediapipe_builtin.defs import PoseLandmarkerPath, HandLandmarkerPath
from src.data.data_reader import DataHandler
from src.features.extract import extract, FEATURE_DIM, FEATURE_VERSION
from src.features.frame import FRAME_SIZE, frame_padding

# ============================ CẤU HÌNH THU THẬP ============================
# Lớp cần thu thập -> số mẫu MỚI cần quay thêm cho lớp đó (lớp chưa có sẽ tự được tạo).
CLASS_PLAN = {}
cls = input("Nhap class: ")
sample = int(input("So sample"))
CLASS_PLAN[cls]=sample

CAMERA_INDEX = 0
MIRROR_DATA = False        # lật gương ảnh TRƯỚC khi trích xuất. PHẢI giống hệt lúc chạy thực tế (inference)!
MIRROR_PREVIEW = True      # chỉ lật hình hiển thị cho dễ tập, KHÔNG ảnh hưởng dữ liệu lưu
COUNTDOWN_SECONDS = 1.0
RECORD_SECONDS = 2.0       # thời lượng mỗi mẫu
MIN_POSE_COVERAGE = 0.8    # ít nhất 80% frame phải có pose
DISPLAY_SIZE = 720
WINDOW = "VSL collect"

POSE_MODEL_PATH = Path(PoseLandmarkerPath)
HAND_MODEL_PATH = Path(HandLandmarkerPath)
DETECTION_CONF = dict(detection=0.5, presence=0.5, tracking=0.5)

SESSION = datetime.now().strftime("%Y%m%d-%H%M%S")   # mọi mẫu của lần chạy này chung 1 session
print("Session:", SESSION)

Session: 20261118-085804


In [2]:
handler = DataHandler()

assert FEATURE_DIM == DataHandler.FEATURE_DIM, (
    f"Extractor trả {FEATURE_DIM} chiều nhưng schema.FRAME_FEATURE_DIM = {DataHandler.FEATURE_DIM}. "
    "Cập nhật FRAME_FEATURE_DIM trong src/features/schema.py."
)
for p in (POSE_MODEL_PATH, HAND_MODEL_PATH):
    assert p.is_file(), f"Không thấy file model: {p}"


def print_table():
    counts = handler.get_class_counts()
    print(f"{'Lớp':<28}{'Số mẫu':>8}")
    print("-" * 36)
    for name, n in counts.items():
        print(f"{name:<28}{n:>8}")
    print("-" * 36)
    print(f"{len(counts)} lớp, {sum(counts.values())} mẫu")
    if handler.incomplete:
        print("\n[cảnh báo] mẫu dở dang/hỏng bị bỏ qua:")
        for path, why in handler.incomplete:
            print("  ", path, "->", why)


print_table()

Lớp                           Số mẫu
------------------------------------
age                               45
cảm ơn                            45
g                                 45
h                                 15
m                                 45
n                                 45
r                                 45
t                                 45
ten_la                            45
toi                               45
u                                 45
xin chào                          45
địa chỉ                           45
------------------------------------
13 lớp, 555 mẫu


In [3]:
class Detector:
    """Pose + Hand landmarker (chế độ VIDEO, timestamp luôn tăng) -> đặc trưng 1 frame."""

    def __init__(self):
        c = DETECTION_CONF
        self.landmarkers = LandmarkerLoader(
            running_mode=VisionRunningMode.VIDEO,
            pose_model_path=POSE_MODEL_PATH, hand_model_path=HAND_MODEL_PATH,
            detection_confidence=c["detection"], presence_confidence=c["presence"],
            tracking_confidence=c["tracking"])

    def __call__(self, frame_bgr):
        pose_res, hand_res, _ = self.landmarkers.detect(frame_bgr)
        return extract(pose_res, hand_res), pose_res, hand_res

    def close(self):
        self.landmarkers.close()


# ---------------- tiện ích hiển thị ----------------
def ascii_text(s):
    """cv2.putText không vẽ được chữ có dấu -> bỏ dấu khi hiển thị trên cửa sổ."""
    s = s.replace("đ", "d").replace("Đ", "D")
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()


def put(img, text, y, color=(255, 255, 255), scale=0.65):
    cv2.putText(img, text, (14, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img, text, (14, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)


POSE_DRAW = [0, 11, 12, 13, 14, 15, 16]


def draw_overlay(img, pose_res, hand_res):
    h, w = img.shape[:2]
    if pose_res and pose_res.pose_landmarks:
        for i in POSE_DRAW:
            lm = pose_res.pose_landmarks[0][i]
            cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 6, (0, 200, 255), -1)
    if hand_res and hand_res.hand_landmarks:
        for lms in hand_res.hand_landmarks:
            for lm in lms:
                cv2.circle(img, (int(lm.x * w), int(lm.y * h)), 3, (0, 255, 0), -1)


def summarize(feats):
    """(tỉ lệ frame có pose, tỉ lệ có tay trái, tỉ lệ có tay phải)"""
    arr = np.stack(feats)
    pres = np.nan_to_num(arr[:, DataHandler.PRESENCE_COLS])
    return float((~np.isnan(arr).all(axis=1)).mean()), float(pres[:, 0].mean()), float(pres[:, 1].mean())


def quality_ok(feats, summary):
    return (len(feats) >= DataHandler.MIN_FRAMES and summary[0] >= MIN_POSE_COVERAGE
            and (summary[1] > 0 or summary[2] > 0))

In [4]:
def collect_class(cap, det, class_name, n_samples):
    """Thu thập `n_samples` mẫu MỚI cho 1 lớp. Trả về "done" | "next" | "quit"."""
    name = handler.add_class(class_name)
    label = ascii_text(name)
    saved = []                                   # sample_idx đã lưu trong lần chạy này
    total = handler.get_class_counts().get(name, 0)
    state, t0 = "READY", time.time()
    feats, frames, summary, fps_est, msg = [], [], None, 30.0, ""

    def log(text):
        nonlocal msg
        msg = ascii_text(text)
        print(text)

    while len(saved) < n_samples:
        ok, raw = cap.read()
        if not ok:
            raise RuntimeError("Không đọc được frame từ camera")
        if MIRROR_DATA:
            raw = cv2.flip(raw, 1)
        frame = frame_padding(raw)
        if frame.shape != (FRAME_SIZE, FRAME_SIZE, 3):
            raise RuntimeError(f"frame_padding phải trả ảnh 720x720, nhận shape {frame.shape}")
        feat, pose_res, hand_res = det(frame)
        now = time.time()

        if state == "COUNTDOWN" and now - t0 >= COUNTDOWN_SECONDS:
            state, t0, feats, frames = "RECORDING", now, [], []
        if state == "RECORDING":
            feats.append(feat)
            frames.append(frame.copy())
            if now - t0 >= RECORD_SECONDS:
                fps_est = len(feats) / max(now - t0, 1e-6)
                summary = summarize(feats)
                state = "REVIEW"

        # ---------------- hiển thị ----------------
        disp = frame.copy()
        draw_overlay(disp, pose_res, hand_res)
        if MIRROR_PREVIEW:
            disp = cv2.flip(disp, 1)
        put(disp, f"{label}  [{len(saved)}/{n_samples} moi | tong {total}]", 28, (255, 255, 0))
        if state == "READY":
            put(disp, "SPACE: bat dau | U: xoa mau vua luu | N: lop ke | Q: thoat", 58)
        elif state == "COUNTDOWN":
            put(disp, f"{max(0.0, COUNTDOWN_SECONDS - (now - t0)):.1f}", DISPLAY_SIZE // 2, (0, 200, 255), 3.0)
        elif state == "RECORDING":
            cv2.circle(disp, (DISPLAY_SIZE - 30, 30), 12, (0, 0, 255), -1)
            put(disp, f"DANG QUAY {now - t0:.1f}/{RECORD_SECONDS:.1f}s  ({len(feats)} frame)", 58, (0, 0, 255))
        else:  # REVIEW
            good = quality_ok(feats, summary)
            put(disp, f"{len(feats)} frame | pose {summary[0]:.0%} | tay T {summary[1]:.0%} | tay P {summary[2]:.0%}",
                58, (0, 255, 0) if good else (0, 0, 255))
            put(disp, "ENTER: luu | R: quay lai | F: luu du chat luong thap", 88)
        if msg:
            put(disp, msg, DISPLAY_SIZE - 16, (0, 200, 255), 0.6)
        cv2.imshow(WINDOW, disp)
        key = cv2.waitKey(1) & 0xFF

        # ---------------- phím ----------------
        if key == ord("q"):
            return "quit"
        if key == ord("n"):
            return "next"
        if state == "READY":
            if key == 32:
                state, t0, msg = "COUNTDOWN", now, ""
            elif key == ord("u") and saved:
                handler.remove_sample(name, saved.pop())
                total = handler.get_class_counts().get(name, 0)
                log("Đã xoá mẫu vừa lưu")
        elif state == "REVIEW":
            if key == ord("r"):
                state = "READY"
                log("Đã bỏ mẫu vừa quay")
            elif key in (13, 10, ord("f")):
                if key != ord("f") and not quality_ok(feats, summary):
                    log("Chất lượng thấp: bấm F để vẫn lưu, hoặc R để quay lại")
                    continue
                try:
                    idx = handler.write_sample(
                        name, feats, frames, fps=fps_est, session=SESSION,
                        meta={"mirror": MIRROR_DATA, "record_seconds": RECORD_SECONDS,
                              "feature_version": FEATURE_VERSION,
                              "pose_coverage": summary[0], "left_hand": summary[1], "right_hand": summary[2]})
                    saved.append(idx)
                    total = handler.get_class_counts().get(name, 0)
                    log(f"Đã lưu {name}/{idx} ({len(feats)} frame, {fps_est:.1f} fps)")
                except ValueError as e:
                    log(f"Không lưu được: {e}")
                state = "READY"
    return "done"

## Chạy thu thập
Chạy ô dưới, bấm vào cửa sổ OpenCV để nhận phím. Hết số mẫu của một lớp sẽ tự chuyển sang lớp kế tiếp.

In [5]:
cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Không mở được camera {CAMERA_INDEX}")
det = Detector()
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)

try:
    for class_name, n in CLASS_PLAN.items():
        print(f"\n=== {class_name}: quay thêm {n} mẫu ===")
        if collect_class(cap, det, class_name, n) == "quit":
            break
finally:
    cap.release()
    det.close()
    cv2.destroyAllWindows()
    for _ in range(4):
        cv2.waitKey(1)       # để cửa sổ thực sự đóng trên macOS/Windows

print()
print_table()

I0000 00:00:1794967086.168326 12391150 init-domain.cc:134] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1794967086.249070 12391157 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794967086.269302 12391160 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794967086.290224 12391174 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794967086.305281 12391175 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.



=== k: quay thêm 15 mẫu ===


W0000 00:00:1794967086.776515 12391162 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Đã lưu k/0 (26 frame, 12.8 fps)
Đã lưu k/1 (28 frame, 13.9 fps)
Đã lưu k/2 (30 frame, 14.5 fps)
Đã lưu k/3 (30 frame, 14.5 fps)
Đã lưu k/4 (32 frame, 15.5 fps)
Đã lưu k/5 (31 frame, 15.5 fps)
Đã lưu k/6 (31 frame, 15.4 fps)
Đã lưu k/7 (31 frame, 15.5 fps)
Đã lưu k/8 (29 frame, 14.2 fps)
Đã lưu k/9 (31 frame, 15.2 fps)
Đã lưu k/10 (27 frame, 13.0 fps)
Đã lưu k/11 (25 frame, 11.9 fps)
Đã lưu k/12 (24 frame, 11.6 fps)
Đã lưu k/13 (31 frame, 15.1 fps)
Đã lưu k/14 (30 frame, 14.7 fps)

Lớp                           Số mẫu
------------------------------------
age                               45
cảm ơn                            45
g                                 45
h                                 15
k                                 15
m                                 45
n                                 45
r                                 45
t                                 45
ten_la                            45
toi                               45
u                                

## Kiểm tra & dọn dẹp

In [ ]:
# Chất lượng các mẫu vừa quay trong session này
rows = [r for r in handler.quality_report() if r["session"] == SESSION]
print(f"{len(rows)} mẫu trong session {SESSION}")
for r in sorted(rows, key=lambda r: r["pose_coverage"])[:10]:
    print(f"  {r['class']}/{r['sample_id']:<4} frames={r['frames']:<3} pose={r['pose_coverage']:.0%} "
          f"tayT={r['left_hand']:.0%} tayP={r['right_hand']:.0%}")

# Xoá một mẫu xấu:  handler.remove_sample("xin chào", 3)
# Xoá cả một lớp:   handler.remove_class("tên lớp")

15 mẫu trong session 20261118-085804
  k/0    frames=26  pose=100% tayT=1% tayP=72%
  k/1    frames=28  pose=100% tayT=0% tayP=54%
  k/2    frames=30  pose=100% tayT=0% tayP=60%
  k/3    frames=30  pose=100% tayT=0% tayP=71%
  k/4    frames=32  pose=100% tayT=0% tayP=58%
  k/5    frames=31  pose=100% tayT=0% tayP=82%
  k/6    frames=31  pose=100% tayT=0% tayP=90%
  k/7    frames=31  pose=100% tayT=0% tayP=71%
  k/8    frames=29  pose=100% tayT=0% tayP=91%
  k/9    frames=31  pose=100% tayT=0% tayP=83%


: 